# Exercise 5 — add_indicators

`add_indicators` is the integration function: it takes an OHLCV DataFrame and returns a copy enriched with all 9 indicator columns. It is the bridge between market_data.py (Day 89) and the backtester (Day 91) — every downstream day in Section 7 starts with `df = add_indicators(store.load(ticker))`.

In [ ]:
import pandas as pd, math

def _synthetic(n=50):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })
def sma(series, window=20):
    return series.rolling(window=window).mean()
def ema(series, window=20):
    return series.ewm(span=window, adjust=False).mean()
def rsi(series, window=14):
    delta = series.diff()
    gain  = delta.clip(lower=0).rolling(window=window).mean()
    loss  = (-delta.clip(upper=0)).rolling(window=window).mean()
    rs    = gain / loss
    return 100 - (100 / (1 + rs))
def macd(series, fast=12, slow=26, signal=9):
    fast_ema    = ema(series, fast)
    slow_ema    = ema(series, slow)
    macd_line   = fast_ema - slow_ema
    signal_line = ema(macd_line, signal)
    return pd.DataFrame({"macd": macd_line, "signal": signal_line,
                          "histogram": macd_line - signal_line})
def bollinger_bands(series, window=20, num_std=2.0):
    middle = sma(series, window)
    std    = series.rolling(window=window).std()
    return pd.DataFrame({"upper": middle + num_std * std,
                          "middle": middle,
                          "lower": middle - num_std * std})

# ── Exercise: implement add_indicators ───────────────────────────────────────

def add_indicators(df, sma_w=20, ema_w=20, rsi_w=14,
                   macd_fast=12, macd_slow=26, macd_sig=9,
                   bb_w=20, bb_std=2.0):
    """Add all standard indicators to a copy of df.

    Uses df["Close"] as the price series.

    Adds 9 new columns:
        sma_{sma_w}, ema_{ema_w}, rsi_{rsi_w}
        macd, macd_signal, macd_hist
        bb_upper, bb_middle, bb_lower

    Never mutates the input DataFrame.
    """
    # TODO:
    # 1. df = df.copy(); close = df["Close"]
    # 2. df[f"sma_{sma_w}"] = sma(close, sma_w)
    # 3. df[f"ema_{ema_w}"] = ema(close, ema_w)
    # 4. df[f"rsi_{rsi_w}"] = rsi(close, rsi_w)
    # 5. m = macd(close, macd_fast, macd_slow, macd_sig)
    #    df["macd"]        = m["macd"]
    #    df["macd_signal"] = m["signal"]
    #    df["macd_hist"]   = m["histogram"]
    # 6. bb = bollinger_bands(close, bb_w, bb_std)
    #    df["bb_upper"]  = bb["upper"]
    #    df["bb_middle"] = bb["middle"]
    #    df["bb_lower"]  = bb["lower"]
    # 7. return df
    return df.copy()


### Checks

In [ ]:
checks = 0

# 1 — add_indicators returns DataFrame with 9 new columns
try:
    df = _synthetic(n=50)
    enriched = add_indicators(df)
    expected_new = {"sma_20", "ema_20", "rsi_14", "macd", "macd_signal",
                    "macd_hist", "bb_upper", "bb_middle", "bb_lower"}
    actual_new = set(enriched.columns) - set(df.columns)
    assert expected_new == actual_new, f"unexpected cols: {actual_new ^ expected_new}"
    checks += 1; print("✅ 1 add_indicators adds exactly the 9 expected columns")
except Exception as e:
    print("❌ 1:", e)

# 2 — does not mutate the input DataFrame
try:
    df = _synthetic(n=50)
    original_cols = list(df.columns)
    _ = add_indicators(df)
    assert list(df.columns) == original_cols, "input DataFrame was mutated!"
    checks += 1; print("✅ 2 add_indicators does not mutate the input DataFrame")
except Exception as e:
    print("❌ 2:", e)

# 3 — same number of rows, 9 more columns
try:
    df = _synthetic(n=50)
    enriched = add_indicators(df)
    assert len(enriched) == len(df), "row count changed"
    assert len(enriched.columns) == len(df.columns) + 9, \
        f"expected {len(df.columns)+9} cols, got {len(enriched.columns)}"
    checks += 1; print("✅ 3 same rows, exactly 9 more columns")
except Exception as e:
    print("❌ 3:", e)

# 4 — row at index 25 has no NaN in any indicator column
try:
    df = _synthetic(n=50)
    enriched = add_indicators(df)
    indicator_cols = ["sma_20", "ema_20", "rsi_14", "macd", "macd_signal",
                      "macd_hist", "bb_upper", "bb_middle", "bb_lower"]
    row = enriched.iloc[25]
    for col in indicator_cols:
        assert not pd.isna(row[col]), f"{col} is NaN at row 25"
    checks += 1; print("✅ 4 all indicator columns are non-NaN at row 25")
except Exception as e:
    print("❌ 4:", e)

# 5 — bb_upper > bb_middle > bb_lower for all non-NaN rows
try:
    df = _synthetic(n=50)
    enriched = add_indicators(df).dropna(subset=["bb_upper", "bb_lower"])
    assert (enriched["bb_upper"] > enriched["bb_middle"]).all()
    assert (enriched["bb_middle"] > enriched["bb_lower"]).all()
    checks += 1; print("✅ 5 bb_upper > bb_middle > bb_lower for all non-NaN rows")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
